# Practice Workshop 5: Merging and combining data

These exercises practice the same skills as Lecture 5 and Workshop 5 and are at
a similar level of difficulty. They only require what was covered in the
lectures and workshops up to this point. When an exercise needs a function that
was not shown in the lectures, the hint explains how to use it.

Each exercise uses one of the data sets in the course's `data/` folder; the
setup cell defines `DATA_PATH` so that file paths work from this folder.

Parts marked **[Advanced]** are optional and go a bit further than the
workshop. Before running each result, predict what you expect to see.

*The previous, harder versions of these exercises are in the `harder_versions/`
folder.*

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA_PATH = '../../data'

***
# Exercise 1: Output gap and unemployment gap (Okun's law)

*Potential GDP* (`GDPPOT`) is the US Congressional Budget Office's (CBO) estimate
of what the economy would produce at normal capacity. The *output gap* measures
how far actual GDP is from this level. *Okun's law* says that when output is
below potential, unemployment is above its natural rate.

You need four files from the folder `data/FRED`, all with quarterly data:

| File | Variable | Description |
|------|----------|-------------|
| `GDP.csv` | `GDP` | Real GDP. |
| `GDPPOT.csv` | `GDPPOT` | Real potential GDP (CBO estimate). |
| `NROU.csv` | `NROU` | Natural rate of unemployment (CBO estimate), in percent. |
| `FRED_quarterly.csv` | `UNRATE` | Unemployment rate, in percent. |

1.  Load `GDP.csv` and `GDPPOT.csv` and parse `DATE` as dates. For each file,
    report the number of rows and the first and the last date.

    *Hint*: Use `pd.read_csv(..., parse_dates=['DATE'])`. Use `len()` to count
    the rows, and `min()` and `max()` on the `DATE` column.

2.  Merge the GDP and potential GDP data on the `DATE` column, once using an
    *inner join* and once using an *outer join*. How many rows does each result
    have?

    *Hint*: Use
    [`pd.merge()`](https://pandas.pydata.org/docs/reference/api/pandas.merge.html)
    with `on='DATE'` and `how='inner'` or `how='outer'`.

    Look at the first and the last rows of the outer join using `head()` and
    `tail()`. Which values are missing, and why?

    *Hint*: Compare the first and last dates you found in part 1. The CBO also
    publishes estimates of potential GDP for future quarters.

3.  Load `NROU.csv` and `FRED_quarterly.csv`. From `FRED_quarterly.csv`, keep
    only the columns `DATE` and `UNRATE`.

    Merge both into the inner join from part 2, again using inner joins. Your
    final `DataFrame` should have the columns `DATE`, `GDP`, `GDPPOT`, `NROU`
    and `UNRATE`. Which period does it cover?

4.  Compute the output gap and the unemployment gap and store them in new
    columns:
    $$
    \text{Output gap} = 100 \times \frac{\text{GDP} - \text{GDPPOT}}{\text{GDPPOT}}
    \qquad
    \text{Unemployment gap} = \text{UNRATE} - \text{NROU}
    $$
    Which quarter had the most negative output gap?

    *Hint*: Use
    [`sort_values()`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.sort_values.html)
    to sort the rows by the output gap.

5.  Use
    [`corr()`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.corr.html)
    to compute the correlation between the two gaps.

    Create a scatter plot with the output gap on the horizontal axis and the
    unemployment gap on the vertical axis. Each point should represent one
    quarter. Add horizontal and vertical lines at zero. Is Okun's law supported
    by the data?

6.  **[Advanced]** Use
    [`np.polyfit(x, y, 1)`](https://numpy.org/doc/stable/reference/generated/numpy.polyfit.html)
    to fit a straight line through the scatter plot and add it to the figure. By
    how many percentage points does unemployment rise above its natural rate
    when output falls 1% below potential?

***
# Exercise 2: Annual stock returns

In this exercise, you compare the annual returns of five tech stocks with the
S&P 500 index. The files are located in the `data/stockmarket` folder:

| File | Columns |
|------|---------|
| `AAPL.csv`, `META.csv`, `MSFT.csv`, `NVDA.csv`, `TSLA.csv` | `Date`, `Open`, `Close` |
| `SP500.csv` | `Date`, `SP500` |

1.  Write a loop over the five ticker symbols. For each ticker:

    1.  load the file, parse `Date` as dates and set it as the index,
    2.  get the last closing price of each calendar year using
        [`resample()`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.resample.html)
        with the rule `'YE'` and `last()`,
    3.  compute the annual return in percent from these year-end prices using
        `pct_change()`, and
    4.  append the result to a list.

    *Hint*: Select the `Close` column before resampling, so that the result is a
    `Series`.

2.  Combine the five `Series` into one `DataFrame` using
    [`pd.concat()`](https://pandas.pydata.org/docs/reference/api/pandas.concat.html)
    along the columns (`axis=1`). Pass your list of ticker symbols as the `keys`
    argument, so that each column is labeled with its ticker symbol, as in the
    lecture. Display the result. Why are some values missing?

    *Hint*: Look at the first year for which each stock has a return.

3.  Drop all years in which at least one stock has a missing return using
    [`dropna()`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.dropna.html).
    Which years remain?

4.  Compute the average annual return of each stock over the remaining years.
    Which stock had the highest average return? Create a bar chart of the
    averages.

5.  Compute the annual return of the S&P 500 in the same way, and add it as a
    new column to the `DataFrame` of stock returns from part 3.

    *Hint*: Use
    [`join()`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.join.html),
    which matches rows by their index (here: the year-end dates).

    How did the five stocks and the index perform in 2022?

6.  **[Advanced]** For each year, find the stock with the highest return, and
    count how many years each stock was the best.

    *Hint*: `idxmax(axis=1)` returns the column label of the largest value in
    each row. Use
    [`value_counts()`](https://pandas.pydata.org/docs/reference/api/pandas.Series.value_counts.html)
    to count.

***
# Exercise 3: How good are inflation expectations?

The files `SCE_2013.csv`, ..., `SCE_2025.csv` in the folder `data/SCE_10pct`
contain the 10% sample of the
[Survey of Consumer Expectations](https://www.newyorkfed.org/microeconomics/sce)
that you used in Workshop 5. Each row represents one respondent in one survey
month:

| Variable | Description |
|----------|-------------|
| `userid` | Respondent identifier. |
| `date` | First day of the survey month. |
| `female` | Gender indicator: 0 = male, 1 = female. |
| `infl_1y` | Expected inflation over the next 12 months, in percent. |

In Workshop 5, you compared expectations with inflation in the *past*. Here, you
compare the average expectation with the inflation that *actually happened* over
the following 12 months.

1.  Write a loop to load the 13 annual files into a list of `DataFrame`s.
    Parse `date` as dates. Concatenate the list into a single `DataFrame` using
    [`pd.concat()`](https://pandas.pydata.org/docs/reference/api/pandas.concat.html)
    with `ignore_index=True`.

    Report the number of rows and the first and the last survey month.

    *Hint*: Loop over `range(2013, 2026)` and use an f-string to create each
    file name.

2.  Compute the mean expected inflation (`infl_1y`) for each survey month.
    Store the result in a `DataFrame` named `df_expected` with the two columns
    `date` and `expected`.

    *Hint*: Group by `date`, then use
    [`rename()`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.rename.html)
    and `reset_index()` as in Workshop 5.

3.  Load `CPI.csv` from the folder `data/FRED`, parse `DATE` as dates and rename
    the column `DATE` to `date` so that it matches the survey data.

    Compute the *realized* inflation over the following 12 months and store it
    in a new column `realized`:
    $$
    \text{Realized}_t = \left( \frac{CPI_{t+12}}{CPI_t} - 1 \right) \times 100
    $$
    where $CPI_{t+12}$ is the CPI 12 months after month $t$.

    *Hint*: `df['CPI'].shift(-12)` moves the values 12 rows *up*, so that each
    row contains the CPI from 12 months later.

4.  Merge `df_expected` with the realized inflation using `pd.merge()` with a
    *left join* on `date`. For how many survey months is the realized inflation
    missing? Why?

    *Hint*: Use `isna()` and `sum()` to count missing values. Look at the last
    date in `CPI.csv`, and check whether any CPI values are missing.

5.  Create a line plot showing expected and realized inflation over time in a
    single graph, with a legend. When did people's expectations differ most from
    the inflation that actually happened?

6.  **[Advanced]** Compute the forecast error, `expected - realized`, for each
    survey month. What is the mean forecast error? In what percentage of months
    did people over-predict inflation (positive error)?